<a href="https://colab.research.google.com/github/WeegorMartins/customer-decisioning-lab/blob/main/notebooks/02_download_criteo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# 02 — Download do benchmark Criteo

A base é usada para benchmark causal.
As variáveis são anônimas e não serão renomeadas como dados bancários.
O arquivo bruto não será enviado ao GitHub.
```

In [1]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/customer-decisioning-lab"
)
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(RAW_DIR)

Mounted at /content/drive
/content/drive/MyDrive/customer-decisioning-lab/data/raw


In [2]:
CRITEO_URL = (
    "https://go.criteo.net/"
    "criteo-research-uplift-v2.1.csv.gz"
)

CRITEO_PATH = (
    RAW_DIR
    / "criteo-uplift-v2.1.csv.gz"
)

!wget -q -O "$CRITEO_PATH" "$CRITEO_URL"

print("Arquivo:", CRITEO_PATH)
print(
    "Tamanho MB:",
    round(
        CRITEO_PATH.stat().st_size / 1_000_000,
        2
    )
)

200
Arquivo: /content/drive/MyDrive/customer-decisioning-lab/data/raw/criteo-uplift-v2.1.csv.gz
Tamanho MB: 311.42


In [3]:
import pandas as pd

preview = pd.read_csv(
    CRITEO_PATH,
    compression="gzip",
    nrows=5
)

display(preview)
print(preview.columns.tolist())

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure']


In [4]:
USECOLS = [
    "f0", "f1", "f2", "f3",
    "f4", "f5", "f6", "f7",
    "f8", "f9", "f10", "f11",
    "treatment",
    "conversion",
    "visit",
    "exposure"
]

CHUNK_SIZE = 200_000
TARGET_ROWS = 1_500_000

parts = []
current_rows = 0

for chunk in pd.read_csv(
    CRITEO_PATH,
    compression="gzip",
    usecols=USECOLS,
    chunksize=CHUNK_SIZE
):
    parts.append(chunk)
    current_rows += len(chunk)
    print("Linhas lidas:", current_rows)

    if current_rows >= TARGET_ROWS:
        break

criteo = pd.concat(
    parts,
    ignore_index=True
).head(TARGET_ROWS)

print("Formato:", criteo.shape)

Linhas lidas: 200000
Linhas lidas: 400000
Linhas lidas: 600000
Linhas lidas: 800000
Linhas lidas: 1000000
Linhas lidas: 1200000
Linhas lidas: 1400000
Linhas lidas: 1600000
Formato: (1500000, 16)


In [5]:
assert criteo["treatment"].isin([0, 1]).all()
assert criteo["conversion"].isin([0, 1]).all()
assert criteo[USECOLS].notna().all().all()

summary = criteo.agg({
    "treatment": ["mean"],
    "conversion": ["mean"],
    "visit": ["mean"],
    "exposure": ["mean"]
})

display(summary)
print("CRITEO VALIDADO")

,treatment,conversion,visit,exposure
mean,1.0,0.004745,0.025038,0.026358


CRITEO VALIDADO


In [6]:
CRITEO_SAMPLE_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "criteo_sample_1500k.parquet"
)

CRITEO_SAMPLE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

criteo.to_parquet(
    CRITEO_SAMPLE_PATH,
    index=False
)

print(CRITEO_SAMPLE_PATH)

/content/drive/MyDrive/customer-decisioning-lab/data/processed/criteo_sample_1500k.parquet
